# Attention Is All You Need 재구현 실습 - Scaled Dot-Product Attention from Scratch

- Tutorial ID: `mod-attention-paper-lab`
- Tutorial: Attention Is All You Need 재구현 실습
- Section ID: `mod-attn-1`
- Section: Scaled Dot-Product Attention from Scratch


# Integrated Notebook: 01_Scaled_Dot_Product_Attention_from_Scratch

## Original Version
Source: 063_mod-attention-paper-lab_mod-attn-1_Attention_Is_All_You_Need_#Uc7ac#Uad6c#Ud604_#Uc2e4#Uc2b5_-_Scaled_Dot-Product_Attention_from_Scratch.ipynb

# Attention Is All You Need 재구현 실습 - Scaled Dot-Product Attention from Scratch

- Tutorial ID: `mod-attention-paper-lab`
- Tutorial: Attention Is All You Need 재구현 실습
- Section: Scaled Dot-Product Attention from Scratch

In [ ]:
# ============================================================
# 코드 읽는 법 — Scaled Dot-Product Attention from Scratch
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#   3) 미래 토큰을 -inf로 막은 뒤 softmax 확률이 0이 되는지 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)

def softmax(x, axis=-1):
        x = x - np.max(x, axis=axis, keepdims=True)
    return np.exp(x) / np.sum(np.exp(x), axis=axis, keepdims=True)

np.random.seed(7)
T, d_model, d_head = 5, 8, 4
X = np.random.randn(T, d_model)
Wq, Wk, Wv, Wo = [np.random.randn(d_model, d_head)/np.sqrt(d_model) for _ in range(3)] + [np.random.randn(d_head, d_model)/np.sqrt(d_head)]
Q, K, V = X @ Wq, X @ Wk, X @ Wv
scores = Q @ K.T / np.sqrt(d_head)
causal = np.triu(np.full((T,T), -1e9), k=1)
A = softmax(scores + causal)
Y = (A @ V) @ Wo
print('raw scores:
', scores)
print('
causal attention pattern rows sum:', A.sum(axis=1))
print(A)
print('
output shape:', Y.shape)
print('last token attends most to position:', int(np.argmax(A[-1])))

## AI Developed Version 1
Source: 063_Scaled_Dot-Product_Attention_from_Scratch.ipynb

# 📘 Scaled Dot-Product Attention from Scratch

## "Attention Is All You Need" 논문 재구현 실습

---

### 🎯 학습 목표

이 노트북에서는 Transformer의 핵심인 **Scaled Dot-Product Attention**을 밑바닥부터 구현합니다.

1. Attention이 왜 필요한지 직관적으로 이해하기
2. Query, Key, Value의 역할 이해하기
3. Scaled Dot-Product Attention 수식을 코드로 구현하기
4. 각 단계별 텐서의 shape 변화 추적하기

---

### 📖 배경 지식: Attention이란?

우리가 문장을 읽을 때, 모든 단어에 같은 집중도를 두지 않습니다.
예를 들어, "The **cat** sat on the **mat**"에서 "cat"과 "mat"의 관계를 파악할 때,
중간의 "sat on the"보다 "cat"과 "mat"에 더 **주의(attention)**를 기울입니다.

Attention 메커니즘은 이러한 인간의 주의력을 수학적으로 모델링한 것입니다.

### 📐 수식

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

- **Q (Query)**: "내가 찾고 싶은 정보" - 질문하는 쪽
- **K (Key)**: "내가 가지고 있는 정보의 라벨" - 답변의 색인(index)
- **V (Value)**: "실제 정보 내용" - 답변의 본문
- **d_k**: Key 벡터의 차원 수 (스케일링에 사용)

#### 왜 √d_k로 나누나요?

d_k가 커지면 Q·K의 내적값도 커지고, softmax의 입력이 극단적으로 커져서
gradient가 거의 0이 됩니다 (vanishing gradient). 이를 방지하기 위해 스케일링합니다.

## 1단계: 필요한 라이브러리 임포트

PyTorch만 사용하여 Attention을 구현합니다.
numpy는 결과 확인용으로 사용합니다.

In [ ]:
import torch
import torch.nn.functional as F  # softmax 등 유용한 함수 모음
import numpy as np

# 재현성을 위한 시드 고정
# (같은 코드를 실행할 때마다 같은 결과가 나오도록 합니다)
torch.manual_seed(42)

print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")

## 2단계: 입력 데이터 준비하기

### 실제 상황을 생각해봅시다

"나는 고양이를 좋아한다"라는 문장이 있다고 합시다.
이 문장은 4개의 토큰(단어)으로 구성되어 있고,
각 토큰은 임베딩 벡터로 표현됩니다.

- **seq_len (시퀀스 길이)** = 4 (토큰 4개)
- **d_model (임베딩 차원)** = 8 (각 토큰을 8차원 벡터로 표현)
- **d_k (Key/Query 차원)** = 8 (이 예제에서는 d_model과 동일)

> 💡 실제 Transformer에서는 d_model=512, d_k=64 등을 사용하지만,
> 이해를 위해 작은 값을 사용합니다.

In [ ]:
# ============================================================
# 하이퍼파라미터 설정

In [ ]:
seq_len = 4    # 문장의 토큰(단어) 수: "나는", "고양이를", "좋아", "한다"
d_model = 8    # 각 토큰의 임베딩 차원 (토큰을 표현하는 벡터의 길이)
d_k = 8        # Query, Key의 차원 (이 예제에서는 d_model과 동일)
d_v = 8        # Value의 차원

print(f"시퀀스 길이 (토큰 수): {seq_len}")
print(f"임베딩 차원: {d_model}")
print(f"Key/Query 차원: {d_k}")
print(f"Value 차원: {d_v}")

In [ ]:
# ============================================================
# 입력 텐서 생성 (임베딩된 토큰들)

In [ ]:
# 실제로는 Embedding 레이어를 통해 생성되지만,
# 여기서는 랜덤 값으로 시뮬레이션합니다.

# x의 shape: (seq_len, d_model) = (4, 8)
# 각 행이 하나의 토큰을 나타냅니다.
x = torch.randn(seq_len, d_model)

print("입력 텐서 x의 shape:", x.shape)
print("\n입력 텐서 x (각 행 = 하나의 토큰 임베딩):")
print(x)
print("\n[해석]")
print(f"  - {seq_len}개의 토큰이 있고")
print(f"  - 각 토큰은 {d_model}차원 벡터로 표현됩니다")

## 3단계: Q, K, V 생성을 위한 가중치 행렬 만들기

### Q, K, V는 무엇인가요?

도서관 검색에 비유해봅시다:

| 개념 | 도서관 비유 | Attention에서의 역할 |
|------|-----------|--------------------|
| **Query (Q)** | 검색어 | "이 토큰이 다른 토큰에게 묻는 질문" |
| **Key (K)** | 책 제목/태그 | "이 토큰이 가진 정보의 라벨" |
| **Value (V)** | 책 내용 | "이 토큰이 전달할 실제 정보" |

입력 x에서 Q, K, V를 만들기 위해 **선형 변환(Linear Transformation)**을 사용합니다.

$$Q = x \cdot W_Q, \quad K = x \cdot W_K, \quad V = x \cdot W_V$$

같은 입력 x에서 서로 다른 가중치(W_Q, W_K, W_V)를 곱해서
서로 다른 "관점"의 표현을 만드는 것입니다.

In [ ]:
# ============================================================
# 가중치 행렬 생성

In [ ]:
# 이 가중치들은 학습을 통해 최적화됩니다.
# 여기서는 랜덤으로 초기화합니다.

# W_Q: (d_model, d_k) = (8, 8) - 입력을 Query 공간으로 변환
W_Q = torch.randn(d_model, d_k)

# W_K: (d_model, d_k) = (8, 8) - 입력을 Key 공간으로 변환
W_K = torch.randn(d_model, d_k)

# W_V: (d_model, d_v) = (8, 8) - 입력을 Value 공간으로 변환
W_V = torch.randn(d_model, d_v)

print(f"W_Q shape: {W_Q.shape}  (d_model → d_k 변환)")
print(f"W_K shape: {W_K.shape}  (d_model → d_k 변환)")
print(f"W_V shape: {W_V.shape}  (d_model → d_v 변환)")

In [ ]:
# ============================================================
# Q, K, V 계산

In [ ]:
# 행렬 곱셈으로 입력을 각각의 공간으로 변환합니다.

# Q = x @ W_Q : (4, 8) @ (8, 8) = (4, 8)
Q = x @ W_Q  # 각 토큰의 "질문" 벡터
K = x @ W_K  # 각 토큰의 "라벨" 벡터
V = x @ W_V  # 각 토큰의 "내용" 벡터

print(f"Q shape: {Q.shape}  → 각 토큰의 Query 벡터")
print(f"K shape: {K.shape}  → 각 토큰의 Key 벡터")
print(f"V shape: {V.shape}  → 각 토큰의 Value 벡터")

print("\n[핵심 포인트]")
print("Q, K, V 모두 같은 입력 x에서 나왔지만,")
print("서로 다른 가중치를 곱해서 서로 다른 역할을 합니다.")
print("이것이 'Self-Attention'의 'Self'가 의미하는 바입니다!")

## 4단계: Attention Score 계산 (QK^T)

이제 각 토큰이 다른 토큰들에 얼마나 "관심"을 가져야 하는지 계산합니다.

$$\text{Score} = Q \cdot K^T$$

이것은 **내적(dot product)**으로 두 벡터의 유사도를 측정하는 것입니다.

- 내적값이 **크면** → 두 토큰이 **관련성이 높음** (유사한 방향)
- 내적값이 **작으면** → 두 토큰이 **관련성이 낮음** (다른 방향)

In [ ]:
# ============================================================
# Step 4-1: QK^T 계산 (Attention Score)

In [ ]:
# Q @ K^T : (4, 8) @ (8, 4) = (4, 4)
# 결과는 (seq_len, seq_len) 크기의 행렬!
# 각 토큰 쌍 사이의 유사도를 나타냅니다.
scores = Q @ K.T  # K.T는 K의 전치행렬 (transpose)

print(f"Attention Score shape: {scores.shape}")
print(f"  → ({seq_len}, {seq_len}) = 각 토큰 쌍의 유사도")
print()
print("Attention Scores (스케일링 전):")
print(scores)
print()
print("[해석 예시]")
print(f"  scores[0][1] = {scores[0][1]:.4f}")
print(f"  → 토큰 0이 토큰 1에 대해 갖는 관심도 점수")
print(f"  scores[0][0] = {scores[0][0]:.4f}")
print(f"  → 토큰 0이 자기 자신에 대해 갖는 관심도 점수")

## 5단계: 스케일링 (÷ √d_k)

### 왜 스케일링이 필요한가?

d_k가 커지면 내적값의 크기도 커집니다.
softmax에 큰 값이 들어가면, 출력이 극단적(0 또는 1에 가까움)이 됩니다.
이를 **saturation**이라 하며, 학습이 잘 되지 않게 만듭니다.

직관적으로:
- **스케일링 없이**: [100, 1, 0] → softmax → [1.0000, 0.0000, 0.0000] (거의 one-hot)
- **스케일링 후**: [10, 0.1, 0] → softmax → [0.9999, 0.0001, 0.0000] (좀 더 부드러움)

In [ ]:
# ============================================================
# Step 5: 스케일링

In [ ]:
import math

# √d_k 계산
scale_factor = math.sqrt(d_k)
print(f"d_k = {d_k}")
print(f"√d_k = {scale_factor:.4f}")
print()

# 스케일링 적용
scaled_scores = scores / scale_factor

print("스케일링 전 scores:")
print(scores)
print()
print("스케일링 후 scores (÷ √d_k):")
print(scaled_scores)
print()
print("[비교] 값의 범위가 줄어든 것을 확인하세요!")
print(f"  스케일링 전 - 최소: {scores.min():.4f}, 최대: {scores.max():.4f}")
print(f"  스케일링 후 - 최소: {scaled_scores.min():.4f}, 최대: {scaled_scores.max():.4f}")

## 6단계: Softmax로 확률 분포 만들기

스케일링된 점수에 softmax를 적용하여 **확률 분포**로 변환합니다.

$$\text{softmax}(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

softmax의 특징:
- 모든 값이 0~1 사이
- 각 행의 합이 1 (확률 분포)
- 큰 값은 더 크게, 작은 값은 더 작게 (차이 강조)

In [ ]:
# ============================================================
# Step 6: Softmax 적용

In [ ]:
# dim=-1: 마지막 차원(행 방향)을 따라 softmax 적용
# 즉, 각 토큰(행)별로 다른 토큰들에 대한 관심도를 확률로 변환
attention_weights = F.softmax(scaled_scores, dim=-1)

print("Attention Weights (확률 분포):")
print(attention_weights)
print()

# 각 행의 합이 1인지 확인
print("각 행의 합 (모두 1이어야 함):")
print(attention_weights.sum(dim=-1))
print()

print("[해석]")
print(f"  토큰 0의 attention weights: {attention_weights[0].tolist()}")
print(f"  → 토큰 0은 각 토큰(0,1,2,3)에 이만큼의 비율로 '주의'를 기울입니다")
print(f"  → 가장 높은 가중치를 가진 토큰에 가장 많이 집중합니다")

## 7단계: Value에 가중치 적용 (최종 출력)

마지막으로, attention weights를 Value에 곱합니다.

$$\text{Output} = \text{Attention Weights} \times V$$

이것은 **가중 평균(weighted average)**입니다.
각 토큰의 출력은 모든 Value 벡터의 가중 평균이 됩니다.

- 관심도가 높은 토큰의 Value는 많이 반영
- 관심도가 낮은 토큰의 Value는 적게 반영

In [ ]:
# ============================================================
# Step 7: 가중 평균 계산 (최종 출력)

In [ ]:
# attention_weights @ V : (4, 4) @ (4, 8) = (4, 8)
# 결과: 각 토큰의 새로운 표현 (다른 토큰들의 정보를 반영한 표현)
output = attention_weights @ V

print(f"입력 shape:  {x.shape}")
print(f"출력 shape:  {output.shape}")
print(f"  → shape이 동일! 입력과 같은 차원의 새로운 표현이 생성됩니다.")
print()
print("Attention 출력:")
print(output)
print()
print("[핵심]")
print("이 출력은 각 토큰이 문맥(context)을 반영한 새로운 표현입니다.")
print("예: '은행'이라는 단어가 있을 때,")
print("   주변에 '돈'이 있으면 → 금융기관 의미로 표현")
print("   주변에 '나무'가 있으면 → 강둑 의미로 표현")
print("이처럼 같은 단어도 문맥에 따라 다른 표현을 가지게 됩니다!")

## 8단계: 전체 과정을 하나의 함수로 정리

지금까지 배운 모든 단계를 하나의 함수로 만들어봅시다.

In [ ]:
# ============================================================
# Scaled Dot-Product Attention 함수 (전체 통합)

In [ ]:
def scaled_dot_product_attention(Q, K, V, verbose=False):
    """
    Scaled Dot-Product Attention을 수행합니다.
    
    Args:
        Q: Query 텐서, shape = (..., seq_len, d_k)
        K: Key 텐서, shape = (..., seq_len, d_k)
        V: Value 텐서, shape = (..., seq_len, d_v)
        verbose: True이면 중간 과정을 출력합니다
    
    Returns:
        output: Attention 결과, shape = (..., seq_len, d_v)
        attention_weights: 각 토큰 쌍의 가중치, shape = (..., seq_len, seq_len)
    """
    # d_k 추출 (Key 벡터의 차원)
    d_k = K.size(-1)
    
    # Step 1: QK^T 계산 (유사도 점수)
    scores = Q @ K.transpose(-2, -1)
    if verbose:
        print(f"[1] QK^T scores shape: {scores.shape}")
    
    # Step 2: √d_k로 스케일링
    scaled_scores = scores / math.sqrt(d_k)
    if verbose:
        print(f"[2] 스케일링 완료 (÷ √{d_k} = ÷ {math.sqrt(d_k):.2f})")
    
    # Step 3: Softmax로 확률 분포 변환
    attention_weights = F.softmax(scaled_scores, dim=-1)
    if verbose:
        print(f"[3] Softmax 적용 → attention_weights shape: {attention_weights.shape}")
    
    # Step 4: Value에 가중치 적용
    output = attention_weights @ V
    if verbose:
        print(f"[4] 최종 출력 shape: {output.shape}")
    
    return output, attention_weights


# 함수 테스트
print("=" * 50)
print("Scaled Dot-Product Attention 함수 테스트")
print("=" * 50)

output, attn_weights = scaled_dot_product_attention(Q, K, V, verbose=True)
print()
print("Attention Weights:")
print(attn_weights)
print()
print("Output:")
print(output)

## 9단계: 시각화로 이해하기

Attention weights를 히트맵으로 시각화하여 직관적으로 이해해봅시다.

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
# Attention Weights 히트맵 시각화

In [ ]:
# 토큰 이름 (예시)
tokens = ["나는", "고양이를", "좋아", "한다"]

fig, ax = plt.subplots(figsize=(6, 5))

# 히트맵 그리기
im = ax.imshow(attn_weights.detach().numpy(), cmap='Blues', vmin=0, vmax=1)

# 축 레이블 설정
ax.set_xticks(range(len(tokens)))
ax.set_yticks(range(len(tokens)))
ax.set_xticklabels(tokens, fontsize=12)
ax.set_yticklabels(tokens, fontsize=12)

ax.set_xlabel("Key (정보를 제공하는 토큰)", fontsize=12)
ax.set_ylabel("Query (정보를 요청하는 토큰)", fontsize=12)
ax.set_title("Attention Weights 히트맵", fontsize=14)

# 각 셀에 값 표시
for i in range(len(tokens)):
    for j in range(len(tokens)):
        text = ax.text(j, i, f"{attn_weights[i, j]:.3f}",
                       ha="center", va="center", fontsize=10,
                       color="white" if attn_weights[i, j] > 0.5 else "black")

plt.colorbar(im, label="Attention Weight")
plt.tight_layout()
plt.show()

print("\n[해석 방법]")
print("  - 행(row): Query 토큰 (질문하는 쪽)")
print("  - 열(col): Key 토큰 (답변하는 쪽)")
print("  - 색이 진할수록 해당 토큰에 더 많은 attention을 줌")
print("  - 각 행의 합 = 1 (확률 분포)")

## 10단계: 배치(Batch) 처리 추가

실제로는 여러 문장을 동시에 처리합니다.
배치 차원을 추가하여 처리해봅시다.

- **batch_size**: 동시에 처리하는 문장 수
- 텐서 shape: `(batch_size, seq_len, d_model)`

In [ ]:
# ============================================================
# 배치 처리 예시

In [ ]:
batch_size = 2  # 2개의 문장을 동시에 처리

# 배치 입력: (batch_size, seq_len, d_model) = (2, 4, 8)
x_batch = torch.randn(batch_size, seq_len, d_model)
print(f"배치 입력 shape: {x_batch.shape}")
print(f"  → {batch_size}개의 문장, 각 문장 {seq_len}개 토큰, {d_model}차원 임베딩")

# Q, K, V 생성 (배치에도 동일한 가중치 적용)
Q_batch = x_batch @ W_Q  # (2, 4, 8)
K_batch = x_batch @ W_K  # (2, 4, 8)
V_batch = x_batch @ W_V  # (2, 4, 8)

print(f"\nQ_batch shape: {Q_batch.shape}")
print(f"K_batch shape: {K_batch.shape}")
print(f"V_batch shape: {V_batch.shape}")

# Attention 수행
output_batch, attn_weights_batch = scaled_dot_product_attention(
    Q_batch, K_batch, V_batch, verbose=True
)

print(f"\n출력 shape: {output_batch.shape}")
print(f"  → {batch_size}개 문장 × {seq_len}개 토큰 × {d_v}차원")
print(f"\nAttention weights shape: {attn_weights_batch.shape}")
print(f"  → {batch_size}개 문장 × {seq_len}개 토큰 × {seq_len}개 토큰")

## 🔑 핵심 정리

### Scaled Dot-Product Attention의 전체 흐름

```
입력 x → W_Q, W_K, W_V 곱셈 → Q, K, V 생성
        → QK^T (유사도 계산)
        → ÷ √d_k (스케일링)
        → softmax (확률 분포)
        → × V (가중 평균)
        → 출력 (문맥을 반영한 새로운 표현)
```

### Shape 변화 정리

| 단계 | 연산 | Shape |
|------|------|-------|
| 입력 x | - | (seq_len, d_model) |
| Q = x @ W_Q | 선형 변환 | (seq_len, d_k) |
| K = x @ W_K | 선형 변환 | (seq_len, d_k) |
| V = x @ W_V | 선형 변환 | (seq_len, d_v) |
| QK^T | 행렬 곱 | (seq_len, seq_len) |
| softmax(QK^T/√d_k) | 스케일+소프트맥스 | (seq_len, seq_len) |
| output | 행렬 곱 | (seq_len, d_v) |

### 다음 단계

- **노트북 064**: Attention 구현 심화 실습
- **노트북 065**: Causal (Masked) Attention 구현
- **노트북 066**: Multi-Head Attention 완전 구현

## AI Developed Version 2
Source: 01_Scaled_Dot_Product_Attention_from_Scratch.ipynb

# 📚 Notebook 1: Scaled Dot-Product Attention from Scratch

## "Attention Is All You Need" 논문 재구현 실습

---

### 🎯 학습 목표
이 노트북에서는 Transformer의 핵심 메커니즘인 **Scaled Dot-Product Attention**을 처음부터 구현합니다.

### 📋 목차
1. Attention이란 무엇인가?
2. Query, Key, Value의 직관적 이해
3. Dot-Product 계산 이해
4. Scaling의 필요성
5. Softmax를 통한 가중치 계산
6. 최종 Attention 출력
7. 전체 구현 및 시각화

### ⚠️ 사전 지식
- Python 기초 (리스트, 함수, 반복문)
- NumPy 기초 (행렬 연산)
- 간단한 선형대수 (행렬 곱셈, 전치)

---

## 1. Attention이란 무엇인가? 🤔

**Attention 메커니즘**은 간단히 말해 "어떤 정보에 더 집중할지 결정하는 방법"입니다.

### 일상생활 비유
도서관에서 시험 공부를 한다고 상상해보세요:
- 📖 **책장의 모든 책** = 입력 시퀀스의 모든 단어들
- 🔍 **내가 찾고 싶은 주제** = Query (질문)
- 🏷️ **각 책의 제목/태그** = Key (검색 키)
- 📄 **책의 실제 내용** = Value (값)

내가 "머신러닝"이라는 주제를 찾고 있을 때(Query),
각 책의 제목(Key)과 비교해서 관련성이 높은 책에 더 "주목"하고,
그 책들의 내용(Value)을 가져오는 것이 바로 Attention입니다!

### 수식으로 표현하면

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

이 수식을 하나씩 차근차근 구현해 봅시다.

## 2. 환경 설정 및 필요한 라이브러리 불러오기

먼저 필요한 라이브러리를 불러옵니다. PyTorch를 사용하여 텐서 연산을 수행합니다.

In [ ]:
# ============================================================
# 필요한 라이브러리 불러오기

In [ ]:
import torch                  # 딥러닝 프레임워크 (텐서 연산용)
import torch.nn as nn         # 뉴럴 네트워크 모듈
import torch.nn.functional as F  # 활성화 함수 등 유틸리티
import math                   # 수학 함수 (제곱근 등)
import matplotlib.pyplot as plt  # 시각화 (그래프, 히트맵 등)
import numpy as np            # 수치 연산

# 결과 재현을 위한 랜덤 시드 고정
# → 같은 코드를 여러 번 실행해도 같은 결과가 나오도록 합니다
torch.manual_seed(42)
np.random.seed(42)

print("✅ 라이브러리 로딩 완료!")
print(f"PyTorch 버전: {torch.__version__}")

## 3. Query, Key, Value 이해하기

### 3.1 먼저 간단한 예시로 시작해봅시다

"나는 고양이를 좋아한다"라는 문장을 생각해 봅시다.
이 문장은 4개의 토큰(단어)으로 이루어져 있습니다.

각 토큰은 **임베딩 벡터**로 표현됩니다. (숫자의 리스트로 변환)

| 개념 | 설명 | 차원 |
|------|------|------|
| **Query (Q)** | "나는 무엇과 관련이 있지?" 라는 질문 | (seq_len, d_k) |
| **Key (K)** | "나를 이런 키워드로 찾을 수 있어" 라는 태그 | (seq_len, d_k) |
| **Value (V)** | "내가 실제로 전달할 정보는 이거야" 라는 내용 | (seq_len, d_v) |

여기서:
- `seq_len` = 시퀀스 길이 (문장의 단어 수)
- `d_k` = Key/Query 벡터의 차원
- `d_v` = Value 벡터의 차원

In [ ]:
# ============================================================
# 3.2 간단한 예시 데이터 만들기

In [ ]:
# 설정값 정의
seq_len = 4   # 문장의 단어 수 ("나는", "고양이를", "좋아", "한다")
d_k = 8       # Query와 Key의 차원 (각 단어를 8개의 숫자로 표현)
d_v = 8       # Value의 차원

# 랜덤으로 Q, K, V 생성
# 실제로는 입력 임베딩에 가중치 행렬(W_Q, W_K, W_V)을 곱해서 만듭니다.
# 여기서는 이해를 위해 랜덤 값으로 시작합니다.
Q = torch.randn(seq_len, d_k)  # Query: 각 단어가 "무엇을 찾고 있는지"
K = torch.randn(seq_len, d_k)  # Key: 각 단어가 "어떤 키워드를 가지는지"
V = torch.randn(seq_len, d_v)  # Value: 각 단어의 "실제 정보"

print("=" * 50)
print("Query, Key, Value 텐서 생성 완료!")
print("=" * 50)
print(f"\nQuery (Q) shape: {Q.shape}  → ({seq_len}개 단어) x ({d_k}차원)")
print(f"Key   (K) shape: {K.shape}  → ({seq_len}개 단어) x ({d_k}차원)")
print(f"Value (V) shape: {V.shape}  → ({seq_len}개 단어) x ({d_v}차원)")

print(f"\n📌 Q의 첫 번째 행 (첫 번째 단어의 Query 벡터):")
print(f"   {Q[0]}")
print(f"   이 벡터는 '나는'이라는 단어가 다른 단어들과 얼마나 관련있는지 계산하는 데 사용됩니다.")

## 4. Step-by-Step: Scaled Dot-Product Attention 구현

### Step 1: QK^T 계산 (유사도 점수)

첫 번째 단계는 Query와 Key 사이의 **유사도(similarity)**를 계산하는 것입니다.

두 벡터의 내적(dot product)이 클수록 → 두 벡터가 비슷한 방향 → 관련성이 높다!

```
Q × K^T = 각 Query와 각 Key 사이의 유사도 점수 행렬
```

In [ ]:
# ============================================================
# Step 1: Q와 K의 내적 (Dot Product) 계산

In [ ]:
# K.T는 K의 전치행렬(transpose)
# K의 shape: (4, 8) → K^T의 shape: (8, 4)
# Q @ K^T: (4, 8) × (8, 4) = (4, 4)
# → 4x4 행렬: 각 단어 쌍의 유사도 점수

scores = torch.matmul(Q, K.transpose(-2, -1))  # Q × K^T

print("📊 유사도 점수 행렬 (Q × K^T):")
print(f"Shape: {scores.shape}  → ({seq_len}x{seq_len}) 행렬")
print(f"\n{scores}")
print(f"\n해석 예시:")
print(f"  scores[0][0] = {scores[0][0]:.4f} → '나는'과 '나는'의 유사도")
print(f"  scores[0][1] = {scores[0][1]:.4f} → '나는'과 '고양이를'의 유사도")
print(f"  scores[0][2] = {scores[0][2]:.4f} → '나는'과 '좋아'의 유사도")
print(f"  scores[0][3] = {scores[0][3]:.4f} → '나는'과 '한다'의 유사도")

### Step 2: Scaling (스케일링) - 왜 √d_k로 나눌까?

#### 🤔 스케일링을 하지 않으면 어떤 문제가 생길까요?

내적값의 크기는 벡터의 차원(d_k)이 커질수록 함께 커집니다.

**예시로 이해해봅시다:**
- d_k = 4일 때: 내적값 범위 대략 -4 ~ +4
- d_k = 64일 때: 내적값 범위 대략 -16 ~ +16  
- d_k = 512일 때: 내적값 범위 대략 -45 ~ +45

값이 너무 크면 → softmax가 극단적인 값을 출력 → 기울기가 거의 0 → 학습이 안 됨!

따라서 √d_k로 나누어 값의 범위를 안정적으로 유지합니다.

In [ ]:
# ============================================================
# Step 2: 스케일링의 효과를 실험으로 확인해봅시다

In [ ]:
print("🔬 스케일링 효과 실험")
print("=" * 50)

# 다양한 d_k 값에서의 내적값 분포 비교
for test_d_k in [4, 64, 512]:
    test_q = torch.randn(1, test_d_k)
    test_k = torch.randn(100, test_d_k)  # 100개의 key와 비교
    
    raw_scores = torch.matmul(test_q, test_k.T)  # 스케일링 전
    scaled_scores = raw_scores / math.sqrt(test_d_k)  # 스케일링 후
    
    print(f"\nd_k = {test_d_k}:")
    print(f"  스케일링 전 - 평균: {raw_scores.mean():.2f}, 표준편차: {raw_scores.std():.2f}")
    print(f"  스케일링 후 - 평균: {scaled_scores.mean():.2f}, 표준편차: {scaled_scores.std():.2f}")

print(f"\n💡 핵심: 스케일링 후에는 d_k와 관계없이 표준편차가 약 1로 유지됩니다!")

In [ ]:
# ============================================================
# Step 2 (계속): 실제 스케일링 적용

In [ ]:
# 스케일링 팩터: √d_k
scale_factor = math.sqrt(d_k)
print(f"스케일링 팩터 (√d_k): √{d_k} = {scale_factor:.4f}")

# 유사도 점수를 √d_k로 나누기
scaled_scores = scores / scale_factor

print(f"\n스케일링 전 점수:")
print(scores)
print(f"\n스케일링 후 점수:")
print(scaled_scores)
print(f"\n✅ 값의 범위가 줄어든 것을 확인할 수 있습니다!")

### Step 3: Softmax로 가중치 계산

스케일링된 점수에 **softmax** 함수를 적용하여 **확률 분포(가중치)**로 변환합니다.

#### Softmax란?
임의의 숫자들을 0~1 사이의 값으로 바꾸고, 합이 1이 되도록 만드는 함수입니다.

```
softmax(x_i) = e^(x_i) / Σ e^(x_j)
```

**예시:**
- 입력: [2.0, 1.0, 0.1] 
- 출력: [0.659, 0.242, 0.099] (합 = 1.0)

In [ ]:
# ============================================================
# Step 3-1: Softmax 함수 이해하기

In [ ]:
# 먼저 softmax가 어떻게 동작하는지 간단한 예시로 확인
simple_scores = torch.tensor([2.0, 1.0, 0.1])

# 수동으로 softmax 계산해보기
exp_scores = torch.exp(simple_scores)  # e^x 계산
softmax_manual = exp_scores / exp_scores.sum()  # 합으로 나누기

# PyTorch의 softmax 함수 사용
softmax_pytorch = F.softmax(simple_scores, dim=0)

print("📐 Softmax 이해하기")
print("=" * 50)
print(f"입력 점수:     {simple_scores.tolist()}")
print(f"e^x 값:        {exp_scores.tolist()}")
print(f"수동 softmax:  {softmax_manual.tolist()}")
print(f"PyTorch 결과:  {softmax_pytorch.tolist()}")
print(f"합계:          {softmax_pytorch.sum().item():.4f}  (항상 1!)")
print(f"\n💡 가장 큰 값(2.0)이 가장 높은 확률(0.659)을 받았습니다!")

In [ ]:
# ============================================================
# Step 3-2: Attention 가중치 계산

In [ ]:
# scaled_scores의 각 행에 softmax 적용
# dim=-1은 "마지막 차원(열 방향)에 대해 softmax 적용"이라는 뜻
attention_weights = F.softmax(scaled_scores, dim=-1)

print("🎯 Attention 가중치 행렬:")
print(f"Shape: {attention_weights.shape}")
print(f"\n{attention_weights}")

# 각 행의 합이 1인지 확인
print(f"\n각 행의 합 (모두 1이어야 함):")
for i in range(seq_len):
    row_sum = attention_weights[i].sum().item()
    print(f"  단어 {i}의 가중치 합: {row_sum:.4f}")

print(f"\n해석: 각 행은 해당 단어가 다른 모든 단어에 '얼마나 주목하는지'를 나타냅니다.")

### Step 4: Value에 가중치 적용 → 최종 출력

마지막으로 Attention 가중치를 Value에 곱하여 **가중 합(weighted sum)**을 계산합니다.

이렇게 하면 각 단어의 출력은 다른 단어들의 Value를 "관련성에 비례하여" 합친 결과가 됩니다.

```
output = attention_weights × V
(4, 4) × (4, 8) = (4, 8)
```

In [ ]:
# ============================================================
# Step 4: 가중 합 계산 → 최종 Attention 출력

In [ ]:
# Attention 가중치 × Value = 최종 출력
output = torch.matmul(attention_weights, V)

print("📦 최종 Attention 출력:")
print(f"Shape: {output.shape}  → ({seq_len}개 단어) x ({d_v}차원)")
print(f"\n{output}")

print(f"\n💡 각 단어의 출력 벡터는 모든 단어의 Value를 가중 평균한 결과입니다.")
print(f"   관련성이 높은 단어의 Value는 더 많이 반영되고,")
print(f"   관련성이 낮은 단어의 Value는 덜 반영됩니다.")

## 5. 전체 함수로 합치기

위의 모든 단계를 하나의 깔끔한 함수로 합쳐봅시다.

In [ ]:
# ============================================================
# Scaled Dot-Product Attention 전체 구현

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Scaled Dot-Product Attention 계산
    
    수식: Attention(Q, K, V) = softmax(QK^T / √d_k) V
    
    Args:
        Q (Tensor): Query 텐서, shape = (..., seq_len, d_k)
        K (Tensor): Key 텐서, shape = (..., seq_len, d_k)
        V (Tensor): Value 텐서, shape = (..., seq_len, d_v)
        mask (Tensor, optional): 마스크 텐서. 마스킹할 위치에 True/1
    
    Returns:
        output (Tensor): Attention 출력, shape = (..., seq_len, d_v)
        attention_weights (Tensor): Attention 가중치, shape = (..., seq_len, seq_len)
    """
    # d_k: Key 벡터의 차원 (마지막 차원의 크기)
    d_k = K.size(-1)
    
    # Step 1: Q와 K^T의 내적으로 유사도 점수 계산
    scores = torch.matmul(Q, K.transpose(-2, -1))
    
    # Step 2: √d_k로 스케일링
    scores = scores / math.sqrt(d_k)
    
    # Step 2.5 (선택): 마스크 적용 (마스킹된 위치에 -inf를 넣어 softmax 후 0이 되도록)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    # Step 3: Softmax로 가중치 계산
    attention_weights = F.softmax(scores, dim=-1)
    
    # Step 4: 가중치 × Value = 최종 출력
    output = torch.matmul(attention_weights, V)
    
    return output, attention_weights

# 함수 테스트
output, weights = scaled_dot_product_attention(Q, K, V)
print("✅ scaled_dot_product_attention 함수 구현 완료!")
print(f"출력 shape: {output.shape}")
print(f"가중치 shape: {weights.shape}")

## 6. Attention 가중치 시각화 🎨

Attention 가중치를 히트맵으로 시각화하면 각 단어가 어떤 단어에 주목하는지 직관적으로 볼 수 있습니다.

In [ ]:
# ============================================================
# Attention 가중치 시각화

In [ ]:
def visualize_attention(attention_weights, tokens=None, title="Attention Weights"):
    """
    Attention 가중치를 히트맵으로 시각화합니다.
    
    Args:
        attention_weights: (seq_len, seq_len) 크기의 가중치 텐서
        tokens: 각 위치에 해당하는 토큰 이름 리스트
        title: 그래프 제목
    """
    weights_np = attention_weights.detach().numpy()
    
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(weights_np, cmap='Blues', vmin=0, vmax=1)
    
    if tokens is None:
        tokens = [f'Token {i}' for i in range(len(weights_np))]
    
    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tokens, fontsize=12)
    ax.set_yticklabels(tokens, fontsize=12)
    ax.set_xlabel('Key (주목 대상)', fontsize=14)
    ax.set_ylabel('Query (주목 주체)', fontsize=14)
    ax.set_title(title, fontsize=16, fontweight='bold')
    
    # 각 셀에 값 표시
    for i in range(len(tokens)):
        for j in range(len(tokens)):
            text = ax.text(j, i, f'{weights_np[i, j]:.3f}',
                          ha='center', va='center',
                          color='white' if weights_np[i, j] > 0.5 else 'black',
                          fontsize=11)
    
    plt.colorbar(im, ax=ax, label='Attention Weight')
    plt.tight_layout()
    plt.show()

# 한국어 토큰 라벨
tokens = ['나는', '고양이를', '좋아', '한다']

# 시각화
visualize_attention(weights, tokens, "Scaled Dot-Product Attention 가중치")

## 7. 배치 처리 지원 (Batch Processing)

실제로는 여러 문장을 동시에 처리합니다. 배치 차원을 추가해봅시다.

```
단일 문장: Q shape = (seq_len, d_k)
배치 처리: Q shape = (batch_size, seq_len, d_k)
```

In [ ]:
# ============================================================
# 배치 처리 테스트

In [ ]:
batch_size = 2  # 동시에 처리할 문장 수
seq_len = 4     # 각 문장의 단어 수
d_k = 8         # Query/Key 차원
d_v = 8         # Value 차원

# 배치 차원이 포함된 Q, K, V
Q_batch = torch.randn(batch_size, seq_len, d_k)
K_batch = torch.randn(batch_size, seq_len, d_k)
V_batch = torch.randn(batch_size, seq_len, d_v)

print(f"배치 Q shape: {Q_batch.shape}  → (배치={batch_size}, 시퀀스={seq_len}, 차원={d_k})")

# 우리의 함수는 배치 처리도 자동으로 지원합니다!
# (PyTorch의 matmul이 배치 차원을 자동 처리)
output_batch, weights_batch = scaled_dot_product_attention(Q_batch, K_batch, V_batch)

print(f"배치 출력 shape: {output_batch.shape}")
print(f"배치 가중치 shape: {weights_batch.shape}")
print(f"\n✅ 배치 처리도 정상 동작합니다!")

## 8. PyTorch nn.Module로 만들기

실제 모델에서 사용할 수 있도록 PyTorch의 `nn.Module`을 상속하여 클래스로 만들어봅시다.

이렇게 하면 입력 임베딩으로부터 Q, K, V를 만드는 선형 변환(Linear Projection)도 포함할 수 있습니다.

In [ ]:
# ============================================================
# nn.Module 기반 Attention 클래스

In [ ]:
class ScaledDotProductAttentionModule(nn.Module):
    """
    Scaled Dot-Product Attention을 PyTorch 모듈로 구현합니다.
    
    입력 임베딩 → Linear 변환 → Q, K, V 생성 → Attention 계산
    """
    
    def __init__(self, d_model, d_k, d_v):
        """
        Args:
            d_model (int): 입력 임베딩 차원 (모델의 기본 차원)
            d_k (int): Query/Key 벡터 차원
            d_v (int): Value 벡터 차원
        """
        super().__init__()
        
        # 입력을 Q, K, V로 변환하는 선형 레이어
        # d_model 차원의 입력을 각각 d_k, d_k, d_v 차원으로 변환
        self.W_Q = nn.Linear(d_model, d_k, bias=False)  # Query 생성용
        self.W_K = nn.Linear(d_model, d_k, bias=False)  # Key 생성용
        self.W_V = nn.Linear(d_model, d_v, bias=False)  # Value 생성용
        
        self.d_k = d_k
    
    def forward(self, x, mask=None):
        """
        Args:
            x (Tensor): 입력 텐서, shape = (batch_size, seq_len, d_model)
            mask (Tensor, optional): 마스크 텐서
        
        Returns:
            output: Attention 출력
            attention_weights: Attention 가중치
        """
        # Step 0: 입력 x로부터 Q, K, V 생성
        Q = self.W_Q(x)  # (batch, seq_len, d_model) → (batch, seq_len, d_k)
        K = self.W_K(x)  # (batch, seq_len, d_model) → (batch, seq_len, d_k)
        V = self.W_V(x)  # (batch, seq_len, d_model) → (batch, seq_len, d_v)
        
        # Step 1~4: Scaled Dot-Product Attention 계산
        output, attention_weights = scaled_dot_product_attention(Q, K, V, mask)
        
        return output, attention_weights

# 테스트
d_model = 16  # 입력 임베딩 차원
d_k = 8       # Query/Key 차원
d_v = 8       # Value 차원

attention_module = ScaledDotProductAttentionModule(d_model, d_k, d_v)

# 가상의 입력 (배치 2, 시퀀스 길이 4, 임베딩 차원 16)
x_input = torch.randn(2, 4, d_model)
output, weights = attention_module(x_input)

print(f"입력 shape: {x_input.shape}")
print(f"출력 shape: {output.shape}")
print(f"가중치 shape: {weights.shape}")
print(f"\n✅ nn.Module 기반 Attention 클래스 완성!")

## 📝 핵심 정리

### Scaled Dot-Product Attention의 핵심 4단계:

| 단계 | 연산 | 설명 |
|------|------|------|
| 1 | Q × K^T | Query와 Key의 유사도 계산 |
| 2 | ÷ √d_k | 값의 크기를 안정화 |
| 3 | softmax | 확률 분포(가중치)로 변환 |
| 4 | × V | 가중 합으로 최종 출력 생성 |

### 다음 노트북 예고 🔜
다음 노트북에서는 이 Attention에 **마스킹(Masking)**을 추가하여
Causal Attention (Decoder에서 사용)을 구현합니다!

## AI Developed Version 3
Source: 063_Scaled_Dot_Product_Attention_from_Scratch.ipynb

# 📘 실습 063: Scaled Dot-Product Attention from Scratch

## 🎯 이 노트북에서 배울 것

**Attention(어텐션)** 은 Transformer 모델의 핵심 아이디어입니다.
"어떤 단어를 볼 때, 다른 어떤 단어들에 집중해야 할까?"를 계산하는 방법입니다.

예를 들어:
```
문장: "그 은행은 강가에 있다"
```
- '은행'이라는 단어가 금융기관인지, 강둑인지 이해하려면
- '강가'라는 단어에 **집중(attention)** 해야 합니다!

---

## 📐 Scaled Dot-Product Attention 공식

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

### 각 기호의 의미:
| 기호 | 이름 | 역할 |
|------|------|------|
| **Q** | Query (쿼리) | "나는 무엇을 찾고 있나?" |
| **K** | Key (키) | "나는 어떤 정보를 갖고 있나?" |
| **V** | Value (값) | "실제로 전달할 정보" |
| **d_k** | 차원 수 | 스케일링을 위한 값 |

### 🏪 도서관 비유
- Q (Query) = 도서관에 검색하는 **검색어** ("파이썬 입문")
- K (Key) = 책에 붙어있는 **키워드 태그** ("파이썬", "프로그래밍" 등)
- V (Value) = 실제 **책의 내용**
- Attention = 검색어와 태그가 얼마나 일치하는지 → 그 비율로 책 내용 가져오기

---

### 학습 목표
1. Attention의 직관적인 의미 이해
2. Q, K, V 행렬 연산 직접 구현
3. softmax 스케일링이 왜 필요한지 이해
4. 간단한 예제로 attention 결과 시각화

In [ ]:
# ============================================================
# 📦 필요한 라이브러리 설치 및 임포트

In [ ]:
# Google Colab에서 실행 시 아래 주석을 해제하고 실행하세요
# !pip install torch matplotlib seaborn

import torch                          # 딥러닝 프레임워크 PyTorch
import torch.nn as nn                # 신경망 구성 요소 (레이어 등)
import torch.nn.functional as F      # 활성화 함수, softmax 등
import math                          # sqrt 등 수학 함수
import matplotlib.pyplot as plt      # 그래프 그리기
import seaborn as sns                # 히트맵 시각화
import numpy as np                   # 수치 계산

# 랜덤 시드 고정 (실행할 때마다 같은 결과가 나오도록)
torch.manual_seed(42)
np.random.seed(42)

print("✅ 라이브러리 임포트 완료!")
print(f"PyTorch 버전: {torch.__version__}")

# GPU 사용 가능 여부 확인
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 중인 디바이스: {device}")

## 🔢 Step 1: 아주 간단한 예제로 시작하기

복잡한 코드 전에, 숫자로 직접 계산해봅시다.

### 예제 문장: "나는 사과를 좋아한다"
각 단어를 숫자 벡터로 표현한다고 가정합니다 (임베딩).

```
나는    → [1.0, 0.0]
사과를  → [0.0, 1.0]  
좋아한다 → [0.5, 0.5]
```

이 3개의 단어가 서로 얼마나 관련되어 있는지를 계산하는 것이 Attention입니다!

In [ ]:
# ============================================================
# 🔢 Step 1: 간단한 예제로 Attention 이해하기

In [ ]:
print("=" * 60)
print("예제: '나는 사과를 좋아한다' 문장의 단어 벡터")
print("=" * 60)

# 각 단어를 2차원 벡터로 표현 (실제로는 512차원 등 더 큰 벡터 사용)
# shape: (단어 수=3, 임베딩 차원=2)
words = ["나는", "사과를", "좋아한다"]

# 단어 임베딩 (단순화된 예시)
embeddings = torch.tensor([
    [1.0, 0.0],   # "나는"
    [0.0, 1.0],   # "사과를"
    [0.5, 0.5],   # "좋아한다"
], dtype=torch.float32)

print(f"\n임베딩 행렬 shape: {embeddings.shape}")
print("(3개 단어 × 2차원 벡터)\n")

for word, emb in zip(words, embeddings):
    print(f"  '{word}': {emb.tolist()}")

In [ ]:
# ============================================================
# 🔢 Step 2: Q, K, V 행렬 만들기

In [ ]:
# 실제 Transformer에서는 학습 가능한 가중치 행렬(W_Q, W_K, W_V)을 통해
# 입력 임베딩으로부터 Q, K, V를 생성합니다.
#
# 여기서는 간단히:
# - Q (Query): 각 단어가 "무엇을 찾고 싶어하는가"
# - K (Key):   각 단어가 "어떤 정보를 갖고 있는가"
# - V (Value): 각 단어의 실제 정보 내용

d_k = 2  # 키 벡터의 차원 (d_k)

# 아주 간단한 가중치 행렬 (실제로는 랜덤 초기화 후 학습됨)
# shape: (임베딩 차원=2, d_k=2)
W_Q = torch.tensor([[1.0, 0.0], [0.0, 1.0]])  # Q 변환 행렬
W_K = torch.tensor([[1.0, 0.0], [0.0, 1.0]])  # K 변환 행렬
W_V = torch.tensor([[1.0, 0.0], [0.0, 1.0]])  # V 변환 행렬

# 임베딩에 가중치 행렬 곱해서 Q, K, V 계산
# shape: (단어 수=3, d_k=2)
Q = embeddings @ W_Q  # (3, 2) @ (2, 2) = (3, 2)
K = embeddings @ W_K  # (3, 2) @ (2, 2) = (3, 2)
V = embeddings @ W_V  # (3, 2) @ (2, 2) = (3, 2)

print("=" * 60)
print("Q (Query) 행렬 - 각 단어가 '무엇을 찾는가':")
print(Q)
print(f"\nK (Key) 행렬 - 각 단어가 '어떤 정보를 갖나':")
print(K)
print(f"\nV (Value) 행렬 - 각 단어의 실제 정보:")
print(V)

In [ ]:
# ============================================================
# 🔢 Step 3: Attention Score 계산

In [ ]:
# Attention Score = QK^T / sqrt(d_k)
#
# QK^T: Query와 Key의 내적(dot product)
#   → 두 벡터가 비슷할수록 값이 크다 = 서로 관련성이 높다!
#
# sqrt(d_k)로 나누는 이유:
#   → d_k가 클수록 내적 값이 커져서 gradient vanishing 문제 발생
#   → 나누면 값을 안정적으로 만들 수 있음

print("=" * 60)
print("Step 3-1: QK^T 계산 (각 단어 쌍의 유사도)")
print("=" * 60)

# QK^T: shape (3, 2) @ (2, 3) = (3, 3)
# [i][j]번째 값 = i번째 단어의 Query와 j번째 단어의 Key의 유사도
scores_before_scale = Q @ K.T
print(f"\nQK^T (스케일링 전):")
print(scores_before_scale)
print(f"\n해석:")
for i, word_i in enumerate(words):
    for j, word_j in enumerate(words):
        score = scores_before_scale[i][j].item()
        print(f"  '{word_i}' → '{word_j}': 유사도 = {score:.2f}")

print("\n" + "=" * 60)
print("Step 3-2: sqrt(d_k)로 스케일링")
print("=" * 60)

# sqrt(d_k)로 나누어 스케일 조정
scores_scaled = scores_before_scale / math.sqrt(d_k)
print(f"\nsqrt(d_k) = sqrt({d_k}) = {math.sqrt(d_k):.4f}")
print(f"\nQK^T / sqrt(d_k) (스케일링 후):")
print(scores_scaled)

In [ ]:
# ============================================================
# 🔢 Step 4: Softmax로 가중치(확률)로 변환

In [ ]:
# Softmax: 점수들을 0~1 사이의 확률 분포로 변환
# 모든 값의 합 = 1
#
# 예: [-0.3, 0.7, 0.2] → softmax → [0.2, 0.5, 0.3]
#                                    합 = 1.0
#
# dim=-1: 마지막 차원(각 행)에 대해 softmax 적용
# 즉, 각 단어(행)에서 다른 모든 단어들에 대한 attention 비율 계산

print("=" * 60)
print("Step 4: Softmax로 Attention Weight 계산")
print("=" * 60)

# softmax 적용
attention_weights = F.softmax(scores_scaled, dim=-1)

print("\nAttention Weights (각 행의 합 = 1.0):")
print(attention_weights)
print()

# 각 단어가 어디에 주목하는지 출력
for i, word_i in enumerate(words):
    print(f"'{word_i}'이(가) 주목하는 비율:")
    for j, word_j in enumerate(words):
        weight = attention_weights[i][j].item()
        bar = "█" * int(weight * 20)  # 시각화
        print(f"  → '{word_j}': {weight:.3f} {bar}")
    print()

# 검증: 각 행의 합이 1.0인지 확인
print(f"각 행의 합 (1.0이어야 함): {attention_weights.sum(dim=-1).tolist()}")

In [ ]:
# ============================================================
# 🔢 Step 5: Value 가중 평균으로 최종 Output 계산

In [ ]:
# Output = Attention_Weights @ V
#
# Attention Weight가 높은 단어의 Value를 더 많이 가져온다!
# → 가중 평균(Weighted Average)
#
# 예: '좋아한다' 단어는 '사과를'에 많이 주목한다면,
#     '사과를'의 Value 정보를 많이 가져와서 표현

print("=" * 60)
print("Step 5: Value 가중 평균으로 최종 Output 계산")
print("=" * 60)

# Output = Attention_Weights @ V
# shape: (3, 3) @ (3, 2) = (3, 2)
output = attention_weights @ V

print("\n입력 임베딩:")
for word, emb in zip(words, embeddings):
    print(f"  '{word}': {emb.tolist()}")

print("\nAttention 적용 후 출력 (문맥 정보 반영됨):")
for word, out in zip(words, output):
    print(f"  '{word}': {[round(x, 3) for x in out.tolist()]}")

print("\n💡 주목: 각 단어의 표현이 문맥을 반영해서 변화했습니다!")

## 🏗️ Step 6: Scaled Dot-Product Attention 함수로 구현

위에서 단계별로 계산한 것을 **함수 하나**로 만들어봅니다.
이것이 바로 "Scaled Dot-Product Attention"입니다!

```
입력: Q, K, V (그리고 선택적으로 mask)
출력: Context Vector (문맥 정보가 담긴 벡터)

과정:
1. scores = Q @ K.T / sqrt(d_k)   ← 유사도 계산
2. weights = softmax(scores)        ← 확률로 변환
3. output = weights @ V            ← 가중 평균
```

In [ ]:
# ============================================================
# 🏗️ Step 6: Scaled Dot-Product Attention 함수 구현

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Scaled Dot-Product Attention 계산
    
    논문: "Attention Is All You Need" (Vaswani et al., 2017)
    공식: Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) * V
    
    Args:
        Q: Query 행렬  shape = (..., seq_len_q, d_k)
           - '나는 무엇을 찾고 있나?'에 해당하는 벡터들
        K: Key 행렬    shape = (..., seq_len_k, d_k)
           - '나는 어떤 정보를 갖고 있나?'에 해당하는 벡터들
        V: Value 행렬  shape = (..., seq_len_k, d_v)
           - 실제로 전달할 정보 벡터들
        mask: 마스크 행렬 (선택적)
              - True/1인 위치를 무시 (미래 토큰 또는 패딩 처리)
    
    Returns:
        output:  shape = (..., seq_len_q, d_v)
                 각 Query 위치에서 Value들의 가중 평균
        weights: shape = (..., seq_len_q, seq_len_k)
                 Attention 가중치 (시각화/분석용)
    """
    
    # d_k: Key 벡터의 차원 수 (스케일링에 사용)
    d_k = Q.size(-1)

In [ ]:
# Step 1: 유사도 계산

In [ ]:
# Q와 K의 내적(dot product) 계산
    # shape: (..., seq_len_q, seq_len_k)
    # [i][j] = i번째 Query와 j번째 Key의 유사도
    scores = torch.matmul(Q, K.transpose(-2, -1))
    
    # sqrt(d_k)로 나누어 스케일 조정
    # 🔑 이 스케일링이 없으면:
    #   - d_k가 클수록 내적 값이 매우 커짐
    #   - softmax가 거의 0 또는 1에 치우침 (기울기 소실)
    #   - 학습이 불안정해짐
    scores = scores / math.sqrt(d_k)

In [ ]:
# Step 2: 마스크 적용 (선택적)

In [ ]:
# mask가 주어진 경우, 마스킹된 위치를 매우 작은 값으로 설정
    # → softmax 후 거의 0이 되어 해당 위치를 무시
    if mask is not None:
        # -1e9 = 매우 작은 수 (softmax 후 ≈ 0이 됨)
        scores = scores.masked_fill(mask == 1, -1e9)

In [ ]:
# Step 3: Softmax로 확률 변환

In [ ]:
# 각 Query에 대해, 모든 Key들의 점수를 확률 분포로 변환
    # dim=-1: 마지막 차원(seq_len_k)에 대해 softmax 적용
    attention_weights = F.softmax(scores, dim=-1)

In [ ]:
# Step 4: Value 가중 평균

In [ ]:
# Attention Weight가 높은 위치의 Value를 더 많이 가져옴
    # shape: (..., seq_len_q, d_v)
    output = torch.matmul(attention_weights, V)
    
    return output, attention_weights


# 위에서 만든 예제로 테스트
print("=" * 60)
print("함수 테스트: 앞서 만든 Q, K, V로 검증")
print("=" * 60)

output_fn, weights_fn = scaled_dot_product_attention(Q, K, V)

print("\n[함수 결과] Attention Weights:")
print(weights_fn.round(decimals=4))
print("\n[함수 결과] Output:")
print(output_fn.round(decimals=4))

print("\n✅ 앞서 단계별로 계산한 결과와 동일!")

In [ ]:
# ============================================================
# 📊 Step 7: Attention Weight 시각화

In [ ]:
print("Attention Weight 히트맵 시각화")
print("(밝을수록 더 많이 주목하는 관계)")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 왼쪽: 간단한 예제 ---
ax1 = axes[0]
sns.heatmap(
    weights_fn.detach().numpy(),
    annot=True,           # 값 표시
    fmt='.3f',            # 소수점 3자리
    cmap='YlOrRd',        # 색상 맵 (노랑→주황→빨강)
    xticklabels=words,    # X축: Key (어디에 주목?
    yticklabels=words,    # Y축: Query (누가 주목?)
    ax=ax1,
    vmin=0, vmax=1        # 0~1 사이 값
)
ax1.set_title('Attention Weights\n(간단한 예제)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Key (어디에 주목?)')
ax1.set_ylabel('Query (누가 주목?)')

# --- 오른쪽: 더 긴 문장 예제 ---
# "The cat sat on the mat" 예제 (영어, 더 직관적)
words_en = ["The", "cat", "sat", "on", "mat"]
seq_len = len(words_en)
d_model = 4

torch.manual_seed(0)
Q_en = torch.randn(seq_len, d_model)
K_en = torch.randn(seq_len, d_model)
V_en = torch.randn(seq_len, d_model)

_, weights_en = scaled_dot_product_attention(Q_en, K_en, V_en)

ax2 = axes[1]
sns.heatmap(
    weights_en.detach().numpy(),
    annot=True,
    fmt='.3f',
    cmap='Blues',
    xticklabels=words_en,
    yticklabels=words_en,
    ax=ax2,
    vmin=0, vmax=1
)
ax2.set_title('Attention Weights\n("The cat sat on mat" 예제)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Key (어디에 주목?)')
ax2.set_ylabel('Query (누가 주목?)')

plt.tight_layout()
plt.savefig('attention_weights_viz.png', dpi=100, bbox_inches='tight')
plt.show()
print("\n💡 히트맵 읽는 법: 행(Query)이 열(Key)에 얼마나 주목하는지를 나타냄")

In [ ]:
# ============================================================
# 🔬 Step 8: 스케일링의 중요성 실험

In [ ]:
#
# 질문: 왜 sqrt(d_k)로 나눠야 할까?
# 실험: 스케일링 유무에 따른 softmax 분포 비교

print("=" * 60)
print("실험: 스케일링이 softmax에 미치는 영향")
print("=" * 60)

# 큰 차원(d_k=512)에서 실험
d_k_large = 512
seq_len_test = 6

torch.manual_seed(42)
Q_test = torch.randn(seq_len_test, d_k_large)
K_test = torch.randn(seq_len_test, d_k_large)

# 스케일링 없는 경우
scores_no_scale = Q_test @ K_test.T
weights_no_scale = F.softmax(scores_no_scale, dim=-1)

# 스케일링 있는 경우
scores_with_scale = Q_test @ K_test.T / math.sqrt(d_k_large)
weights_with_scale = F.softmax(scores_with_scale, dim=-1)

print(f"\nd_k = {d_k_large} 일 때")
print()
print("스케일링 없음:")
print(f"  scores 평균값: {scores_no_scale.mean().item():.2f}")
print(f"  scores 표준편차: {scores_no_scale.std().item():.2f}")
print(f"  softmax 결과 (첫 번째 행): {weights_no_scale[0].tolist()[:6]}")
print(f"  → 값이 0 아니면 1에 가깝게 치우침 (one-hot에 가까움)")
print()
print("스케일링 있음 (/ sqrt(d_k)):")
print(f"  scores 평균값: {scores_with_scale.mean().item():.2f}")
print(f"  scores 표준편차: {scores_with_scale.std().item():.2f}")
print(f"  softmax 결과 (첫 번째 행): {[round(x, 4) for x in weights_with_scale[0].tolist()[:6]]}")
print(f"  → 값이 골고루 분산됨 (더 부드러운 분포)")

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(seq_len_test), weights_no_scale[0].detach().numpy(), color='coral', edgecolor='black')
axes[0].set_title(f'스케일링 없음 (d_k={d_k_large})\n→ 한 곳에 집중, 나머지는 0', fontsize=11)
axes[0].set_xlabel('Key 위치'); axes[0].set_ylabel('Attention Weight')
axes[0].set_ylim(0, 1)

axes[1].bar(range(seq_len_test), weights_with_scale[0].detach().numpy(), color='steelblue', edgecolor='black')
axes[1].set_title(f'스케일링 있음 (÷√{d_k_large})\n→ 여러 곳에 골고루 분산', fontsize=11)
axes[1].set_xlabel('Key 위치'); axes[1].set_ylabel('Attention Weight')
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig('scaling_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n✅ 결론: 스케일링 없이는 gradient vanishing 문제 발생 → 학습 불가!")

In [ ]:
# ============================================================
# 🧪 Step 9: 배치(Batch) 처리 지원 - 실제 사용 형태

In [ ]:
#
# 실제 딥러닝에서는 여러 문장을 동시에 처리합니다 (배치 처리)
# shape: (batch_size, seq_len, d_model)

print("=" * 60)
print("배치(Batch) 처리 테스트")
print("=" * 60)

batch_size = 3      # 한 번에 처리할 문장 수
seq_len = 5         # 각 문장의 단어 수 (= 시퀀스 길이)
d_k = 64            # Query/Key 벡터 차원
d_v = 64            # Value 벡터 차원

torch.manual_seed(42)

# 배치 형태의 Q, K, V 생성
# shape: (batch_size=3, seq_len=5, d_k=64)
Q_batch = torch.randn(batch_size, seq_len, d_k)
K_batch = torch.randn(batch_size, seq_len, d_k)
V_batch = torch.randn(batch_size, seq_len, d_v)

print(f"\n입력 shape:")
print(f"  Q: {Q_batch.shape}  ← (배치=3, 시퀀스=5, d_k=64)")
print(f"  K: {K_batch.shape}  ← (배치=3, 시퀀스=5, d_k=64)")
print(f"  V: {V_batch.shape}  ← (배치=3, 시퀀스=5, d_v=64)")

# 함수 호출 (배치 처리 자동 지원 - torch.matmul은 배치 차원 자동 처리)
output_batch, weights_batch = scaled_dot_product_attention(Q_batch, K_batch, V_batch)

print(f"\n출력 shape:")
print(f"  output:  {output_batch.shape}  ← (배치=3, 시퀀스=5, d_v=64)")
print(f"  weights: {weights_batch.shape}  ← (배치=3, 시퀀스=5, 시퀀스=5)")

# 검증: 각 배치의 각 행에서 softmax 합 = 1.0
sum_check = weights_batch.sum(dim=-1)
print(f"\n검증 - 각 행의 합 (모두 1.0이어야 함):")
print(f"  min = {sum_check.min().item():.6f}")
print(f"  max = {sum_check.max().item():.6f}")
print("\n✅ 배치 처리 완벽하게 동작!")

In [ ]:
# ============================================================
# 🧩 Step 10: nn.Module 클래스로 구현 (PyTorch 스타일)

In [ ]:
#
# 실제 PyTorch 모델에서는 nn.Module을 상속받아 클래스로 구현합니다.
# 이렇게 하면 모델 파라미터가 자동으로 관리되고, GPU 이동 등이 쉬워집니다.

class ScaledDotProductAttention(nn.Module):
    """
    Scaled Dot-Product Attention을 PyTorch nn.Module로 구현
    
    nn.Module을 상속받으면:
    - model.parameters()로 학습 파라미터 추적 가능
    - model.to(device)로 GPU/CPU 이동 쉬움
    - model.eval() / model.train() 모드 전환 가능
    - torch.save()로 모델 저장 가능
    """
    
    def __init__(self, dropout_prob=0.0):
        """
        Args:
            dropout_prob: Dropout 확률 (0.0이면 사용 안 함)
                          과적합 방지를 위해 attention weight에 적용
        """
        super(ScaledDotProductAttention, self).__init__()
        
        # Dropout: 학습 시 일부 attention weight를 0으로 만들어 과적합 방지
        # 논문에서는 attention weight에 dropout 적용 권장
        self.dropout = nn.Dropout(p=dropout_prob)
    
    def forward(self, Q, K, V, mask=None):
        """
        순전파(Forward Pass) 계산
        
        Args:
            Q:    shape = (..., seq_len_q, d_k)
            K:    shape = (..., seq_len_k, d_k)
            V:    shape = (..., seq_len_k, d_v)
            mask: shape = (..., seq_len_q, seq_len_k) or None
        
        Returns:
            output:  shape = (..., seq_len_q, d_v)
            weights: shape = (..., seq_len_q, seq_len_k)
        """
        d_k = Q.size(-1)  # 마지막 차원 = d_k
        
        # 1. 유사도 계산 + 스케일링
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
        
        # 2. 마스크 적용 (선택적)
        if mask is not None:
            scores = scores.masked_fill(mask == 1, float('-inf'))
        
        # 3. Softmax
        weights = F.softmax(scores, dim=-1)
        
        # 4. Dropout (학습 시에만 적용)
        weights = self.dropout(weights)
        
        # 5. Value 가중 평균
        output = torch.matmul(weights, V)
        
        return output, weights


# 모델 인스턴스 생성 및 테스트
attention_layer = ScaledDotProductAttention(dropout_prob=0.1)

# 테스트
batch_size, seq_len, d_k, d_v = 2, 6, 32, 32
torch.manual_seed(42)
Q_test = torch.randn(batch_size, seq_len, d_k)
K_test = torch.randn(batch_size, seq_len, d_k)
V_test = torch.randn(batch_size, seq_len, d_v)

# eval 모드에서 테스트 (dropout 비활성화)
attention_layer.eval()
with torch.no_grad():  # gradient 계산 비활성화
    out, wts = attention_layer(Q_test, K_test, V_test)

print("=" * 60)
print("nn.Module 클래스 테스트 결과")
print("=" * 60)
print(f"입력 Q: {Q_test.shape}")
print(f"출력 output: {out.shape}")
print(f"출력 weights: {wts.shape}")
print(f"\n학습 파라미터 수: {sum(p.numel() for p in attention_layer.parameters())}")
print("(ScaledDotProductAttention 자체에는 학습 파라미터 없음!)")
print("(파라미터는 W_Q, W_K, W_V 행렬에 있으며, 이는 나중에 추가)")

## 📋 정리

### ✅ 이번 실습에서 배운 것

1. **Attention의 직관적 의미**
   - Q: 무엇을 찾고 싶은가 (Query)
   - K: 어떤 정보가 있는가 (Key)
   - V: 실제 전달할 정보 (Value)

2. **Scaled Dot-Product Attention 공식**
   ```
   Attention(Q,K,V) = softmax(QK^T / √d_k) × V
   ```

3. **√d_k 스케일링의 필요성**
   - 큰 d_k에서 내적값이 커져 gradient vanishing 발생
   - 나누면 안정적인 학습 가능

4. **PyTorch nn.Module로 구현하는 방법**

### ➡️ 다음 실습 (064)
- Attention을 직접 구현하는 실습 문제
- 빈 칸 채우기 형태로 직접 코딩해봅니다!